In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
SINKRONISASI KATALOG VENEZUELA DENGAN JSON WAVEFORM
"""
import json
import pandas as pd
import re
import os
from datetime import datetime, timedelta
import numpy as np

# =============================================
# 1. KONFIGURASI PATH
# =============================================

JSON_VENEZUELA_PATH = '/Volumes/Extreme SSD/venezuela_data_earthquake/json_venezuela/extracted_data_3c_venez_v3.json'
CATALOG_VENEZUELA_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/venezuela_earthquake_bench/query_venezuela.csv"
JSON_OUTPUT_PATH = '/Volumes/Extreme SSD/venezuela_data_earthquake/sync_katalog_json/extracted_data_3c_sync_v3.json'

# =============================================
# FUNGSI EKSTRAKSI TIMESTAMP DAN VALIDASI
# =============================================

def extract_timestamp_from_key(key):
    """
    Ekstrak timestamp dari key JSON.
    Format yang diharapkan: NET_STA_YYYYMMDD_HHMMSS
    """
    match = re.search(r'(\d{8})_(\d{6})', key)
    if match:
        return f"{match.group(1)}_{match.group(2)}"
    match = re.search(r'(\d{14})', key)
    if match:
        ts = match.group(1)
        return f"{ts[:8]}_{ts[8:]}"
    return None

def validate_sync_result(json_path):
    print("\n[INFO] Validasi hasil sinkronisasi...")
    with open(json_path, 'r') as f:
        data = json.load(f)
    le_count = 0
    no_count = 0
    other_count = 0
    for key, record in data.items():
        label = record.get('type', '').lower()
        if label in ['le', 'se', 'eq', 'earthquake']:
            le_count += 1
        elif label in ['no', 'noise']:
            no_count += 1
        else:
            other_count += 1
    print(f"   🟢 LE (Earthquake): {le_count}")
    print(f"   🔴 NO (Noise): {no_count}")
    print(f"   ⚪ Lainnya: {other_count}")
    return le_count, no_count

# =============================================
# FUNGSI BACA KATALOG
# =============================================

def read_catalog(csv_path):
    print(f"[INFO] Membaca katalog: {csv_path}")
    df = pd.read_csv(csv_path)
    print(f"   Total baris: {len(df)}")
    print(f"   Kolom: {df.columns.tolist()}")
    
    time_col = None
    for col in df.columns:
        if 'time' in col.lower() or 'datetime' in col.lower() or 'origin' in col.lower():
            time_col = col
            break
    if time_col is None:
        for col in df.columns:
            if df[col].dtype == 'object':
                try:
                    pd.to_datetime(df[col].iloc[0])
                    time_col = col
                    break
                except:
                    continue
    if time_col is None:
        raise ValueError("❌ Tidak dapat menemukan kolom waktu!")
    
    lat_col = next((col for col in df.columns if 'lat' in col.lower()), None)
    lon_col = next((col for col in df.columns if 'lon' in col.lower()), None)
    mag_col = next((col for col in df.columns if 'mag' in col.lower()), None)
    
    label_col = None
    preferred = ['type', 'label', 'class']
    for col in df.columns:
        if col.lower() in preferred:
            label_col = col
            break
    if label_col is None:
        for col in df.columns:
            if 'type' in col.lower() or 'label' in col.lower() or 'class' in col.lower():
                label_col = col
                break
    
    df['datetime'] = pd.to_datetime(df[time_col], utc=True)
    df['timestamp_key'] = df['datetime'].dt.strftime("%Y%m%d_%H%M%S")
    
    metadata = {
        'time_col': time_col,
        'lat_col': lat_col,
        'lon_col': lon_col,
        'mag_col': mag_col,
        'label_col': label_col
    }
    
    print(f"   Kolom waktu: '{time_col}'")
    print(f"   Kolom latitude: '{lat_col}'")
    print(f"   Kolom longitude: '{lon_col}'")
    if mag_col: print(f"   Kolom magnitude: '{mag_col}'")
    if label_col: print(f"   Kolom label: '{label_col}'")
    
    return df, metadata

# =============================================
# FUNGSI SINKRONISASI
# =============================================

def sync_catalog_to_json(json_path, df_catalog, catalog_metadata, output_path):
    print("\n[INFO] Memuat JSON Venezuela...")
    with open(json_path, 'r') as f:
        data = json.load(f)
    print(f"   Total entri di JSON: {len(data)}")
    
    catalog_dict = {}
    for _, row in df_catalog.iterrows():
        key = row['timestamp_key']
        raw_label = row[catalog_metadata['label_col']] if catalog_metadata['label_col'] else 'earthquake'
        if isinstance(raw_label, str):
            lbl = raw_label.lower()
            if 'earthquake' in lbl or 'quake' in lbl:
                mapped_label = 'le'
            else:
                mapped_label = 'no'
        else:
            mapped_label = 'le'
        catalog_dict[key] = {
            'origin_time': row['datetime'].isoformat(),
            'latitude': row[catalog_metadata['lat_col']],
            'longitude': row[catalog_metadata['lon_col']],
            'magnitude': row[catalog_metadata['mag_col']] if catalog_metadata['mag_col'] else None,
            'label': mapped_label
        }
    
    print(f"   Total event di katalog: {len(catalog_dict)}")
    
    matched_count = 0
    unmatched_count = 0
    updated_json = {}
    
    for key, record in data.items():
        ts_str = extract_timestamp_from_key(key)
        if ts_str is None:
            unmatched_count += 1
            record['type'] = 'no'
            updated_json[key] = record
            continue
        
        if ts_str in catalog_dict:
            matched_count += 1
            cat_info = catalog_dict[ts_str]
            record['type'] = cat_info['label']
            if 'metadata' not in record:
                record['metadata'] = {}
            record['metadata']['origin_time'] = cat_info['origin_time']
            record['metadata']['latitude'] = cat_info['latitude']
            record['metadata']['longitude'] = cat_info['longitude']
            if cat_info['magnitude'] is not None:
                record['metadata']['magnitude'] = float(cat_info['magnitude'])
            updated_json[key] = record
        else:
            unmatched_count += 1
            record['type'] = 'no'
            updated_json[key] = record
    
    print(f"\n[INFO] Hasil sinkronisasi:")
    print(f"   ✅ Match: {matched_count}")
    print(f"   ❌ Tidak match: {unmatched_count}")
    
    with open(output_path, 'w') as f:
        json.dump(updated_json, f, indent=2)
    print(f"[SUCCESS] JSON tersimpan di: {output_path}")
    return matched_count, unmatched_count

# =============================================
# MAIN
# =============================================

if __name__ == "__main__":
    print("="*70)
    print("🔗 SINKRONISASI KATALOG VENEZUELA DENGAN JSON WAVEFORM")
    print("="*70)
    
    if not os.path.exists(JSON_VENEZUELA_PATH):
        print(f"❌ ERROR: File JSON tidak ditemukan di: {JSON_VENEZUELA_PATH}")
        exit(1)
    
    if not os.path.exists(CATALOG_VENEZUELA_PATH):
        print(f"❌ ERROR: File katalog tidak ditemukan di: {CATALOG_VENEZUELA_PATH}")
        exit(1)
    
    output_dir = os.path.dirname(JSON_OUTPUT_PATH)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)
        print(f"[INFO] Direktori output dibuat: {output_dir}")
    
    try:
        df_catalog, catalog_metadata = read_catalog(CATALOG_VENEZUELA_PATH)
    except Exception as e:
        print(f"❌ ERROR membaca katalog: {e}")
        exit(1)
    
    matched, unmatched = sync_catalog_to_json(
        JSON_VENEZUELA_PATH,
        df_catalog,
        catalog_metadata,
        JSON_OUTPUT_PATH
    )
    
    le_count, no_count = validate_sync_result(JSON_OUTPUT_PATH)
    
    print("\n" + "="*70)
    print("✅ SINKRONISASI SELESAI!")
    print(f"📂 Output: {JSON_OUTPUT_PATH}")
    print(f"📊 Total event: {matched + unmatched}")
    print(f"   - Earthquake (LE): {le_count}")
    print(f"   - Noise (NO): {no_count}")
    print("="*70)

🔗 SINKRONISASI KATALOG VENEZUELA DENGAN JSON WAVEFORM
[INFO] Membaca katalog: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/venezuela_earthquake_bench/query_venezuela.csv
   Total baris: 745
   Kolom: ['time', 'latitude', 'longitude', 'depth', 'mag', 'magType', 'nst', 'gap', 'dmin', 'rms', 'net', 'id', 'updated', 'place', 'type', 'horizontalError', 'depthError', 'magError', 'magNst', 'status', 'locationSource', 'magSource']
   Kolom waktu: 'time'
   Kolom latitude: 'latitude'
   Kolom longitude: 'longitude'
   Kolom magnitude: 'mag'
   Kolom label: 'type'

[INFO] Memuat JSON Venezuela...
   Total entri di JSON: 3942
   Total event di katalog: 745

[INFO] Hasil sinkronisasi:
   ✅ Match: 3942
   ❌ Tidak match: 0
[SUCCESS] JSON tersimpan di: /Volumes/Extreme SSD/venezuela_data_earthquake/sync_katalog_json/extracted_data_3c_sync_v3.json

[INFO] Validasi hasil sinkronisasi...
   🟢 LE (Earthquake): 3942
   🔴 NO (Noise): 0
   ⚪ Lainnya: 0

✅ SINKRONISASI SELESAI!
📂 Ou